In [ ]:
import requests
import random
from collections import Counter

def get_hot_entities_alternative(k=1000, language='en', month='01'):
    """
    备用方法：通过页面浏览量获取热门实体
    """
    headers = {
            'User-Agent': 'WikidataBatchQuery/1.0 (https://example.com; contact@example.com)'
        }
    view_api_url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/2024/{month}/10"
    # 这里简化处理，实际需要处理日期等参数
    response = requests.get(view_api_url, headers=headers)
    print(response)
    if response.status_code == 200:
        data = response.json()
        articles = data['items'][0]['articles'][:k*2]
        
        wikidata_ids = []
        for article in articles:
            qid = get_wikidata_id_from_title(article['article'], language)
            if qid:
                wikidata_ids.append(qid)
            
            if len(wikidata_ids) >= k:
                break
        
        return wikidata_ids


def get_wikidata_id_from_title(title, language='en'):
    """
    通过维基百科页面标题获取对应的Wikidata ID
    """
    api_url = "https://www.wikidata.org/w/api.php"
    headers = {
            'User-Agent': 'WikidataBatchQuery/1.0 (https://example.com; contact@example.com)'
        }
    
    params = {
        'action': 'wbgetentities',
        'titles': title,
        'sites': f'{language}wiki',
        'props': 'info',
        'format': 'json'
    }
    
    try:
        response = requests.get(api_url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        
        if 'entities' in data:
            for entity_id, entity_data in data['entities'].items():
                if entity_id != '-1':  # 跳过不存在的实体
                    return entity_id
    except requests.RequestException as e:
        print(f"获取Wikidata ID时出错: {e}")
    
    return None

def get_entity_views(qid, days=30):
    """
    获取实体在指定天数内的页面浏览量（需要Wikimedia API token）
    """
    # 注意：这个功能需要申请 Wikimedia API token
    # 这里只是示例代码框架
    pass

# 使用示例
if __name__ == "__main__":
    # 获取热度最高的10个实体的Wikidata ID # 2023/2020
    hot_entities = get_hot_entities_alternative(k=10)
    
    print("热度最高的Wikidata实体:")
    for i, qid in enumerate(hot_entities, 1):
        print(f"{i}. {qid}")
    

In [ ]:
import requests
import json
import time
import re
import concurrent.futures
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
import random

class WikidataTemporalKnowledgeExtractor:
    def __init__(self, delay=0.1, max_workers=5):
        """
        初始化Wikidata时间知识抽取器
        
        参数:
            delay: 请求间延迟（秒）
            max_workers: 最大并行工作线程数
        """
        self.delay = delay
        self.max_workers = max_workers
        self.headers = {
            'User-Agent': 'WikidataTemporalKnowledgeExtractor/2.0'
        }
        
        # 时间限定符属性
        self.time_qualifier_properties = [
            'P585',  # point in time
            'P580',  # start time
            'P582',  # end time
        ]
        
        # 缓存
        self.label_cache = {}
        self.processed_entities = set()
        self.remain_entites = set()
        
        # 批量处理参数
        self.batch_size = 50
        self.entity_priority = {}
        
        # 知识存储
        self.point_knowledge = []
        self.interval_knowledge = []
        self.knowledge_keys = set()
        
    def run_sparql_query(self, query, max_retries=3):
        """
        执行SPARQL查询并返回JSON结果
        """
        params = {
            'format': 'json',
            'query': query
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(
                    "https://query.wikidata.org/sparql", 
                    params=params, 
                    headers=self.headers, 
                    timeout=60
                )
                
                if response.status_code == 200:
                    return response.json()
                elif response.status_code == 429:
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"SPARQL查询HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
            except Exception as e:
                print(f"SPARQL查询异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(5)
        
        return None
    
    def get_entities_batch(self, entity_ids, max_retries=3):
        """
        批量获取实体数据
        """
        if not entity_ids:
            return {}
            
        ids_str = "|".join(entity_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en'
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params, headers=self.headers, timeout=60)
                
                if response.status_code == 200:
                    data = response.json()
                    return data.get('entities', {})
                elif response.status_code == 429:
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"批量获取实体HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
            except Exception as e:
                print(f"批量获取实体异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(5)
        
        return {}
    
    def get_labels_batch(self, entity_ids, max_retries=3):
        """
        批量获取实体标签
        """
        if not entity_ids:
            return {}
            
        q_ids = [eid for eid in entity_ids if eid.startswith('Q') or eid.startswith('P')]
        if not q_ids:
            return {}
            
        ids_str = "|".join(q_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en',
            'props': 'labels'
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params, headers=self.headers, timeout=60)
                
                if response.status_code == 200:
                    data = response.json()
                    entities = data.get('entities', {})
                    
                    labels = {}
                    for eid, entity in entities.items():
                        labels_data = entity.get('labels', {})
                        if 'en' in labels_data:
                            labels[eid] = labels_data['en'].get('value', eid)
                        else:
                            for lang in labels_data:
                                if labels_data[lang].get('value'):
                                    labels[eid] = labels_data[lang].get('value')
                                    break
                            else:
                                labels[eid] = eid
                    
                    self.label_cache.update(labels)
                    return labels
                elif response.status_code == 429:
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                else:
                    #print(f"获取标签HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
            except Exception as e:
                #print(f"获取标签异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(5)
        
        return {}
    
    def extract_entity_id(self, uri):
        """从URI中提取实体ID"""
        if uri and uri.startswith("http://www.wikidata.org/entity/"):
            return uri.split("/")[-1]
        return None
    
    def extract_temporal_knowledge_from_entities(self, entity_batch):
        """
        从实体数据中提取时间知识（使用API方式）
        """
        point_knowledge = []
        interval_knowledge = []
        new_entities = set()
        
        # 批量获取实体数据
        entity_data = self.get_entities_batch(entity_batch)
        if not entity_data:
            return point_knowledge, interval_knowledge, new_entities
        
        # 收集需要标签的实体
        entities_to_label = set()
        
        for entity_id, entity in entity_data.items():
            entities_to_label.add(entity_id)
            
            if 'claims' not in entity:
                continue
                
            # 遍历所有属性声明
            for prop_id, claims in entity['claims'].items():
                entities_to_label.add(prop_id)
                
                for claim in claims:
                    if 'qualifiers' not in claim:
                        continue
                    
                    qualifiers = claim['qualifiers']
                    
                    # 检查时间限定符
                    has_time_qualifiers = any(
                        qual_prop in qualifiers 
                        for qual_prop in self.time_qualifier_properties
                    )
                    
                    if not has_time_qualifiers:
                        continue
                    
                    # 获取主声明值
                    mainsnak = claim.get('mainsnak', {})
                    datavalue = mainsnak.get('datavalue', {})
                    
                    # 提取主声明值信息
                    main_value_id, main_value_type = self._extract_main_value_info(mainsnak)
                    
                    # 只处理实体类型的尾实体
                    if main_value_type != 'entity' or not main_value_id:
                        continue
                    
                    entities_to_label.add(main_value_id)
                    new_entities.add(main_value_id)
                    
                    # 提取时间限定符值
                    time_data = self._extract_time_qualifiers(qualifiers)
                    
                    # 构建知识条目
                    knowledge_base = {
                        'subject': entity_id,
                        'predicate': prop_id,
                        'object': main_value_id,
                        'subject_label': '',
                        'predicate_label': '',
                        'object_label': ''
                    }
                    
                    # 处理时间点知识
                    for time_prop, time_values in time_data['point_times'].items():
                        for time_val in time_values:
                            point_knowledge.append({
                                **knowledge_base,
                                'time_qualifier': time_prop,
                                'time': time_val
                            })
                    
                    # 处理时间区间知识
                    if time_data['interval_times']:
                        for interval in time_data['interval_times']:
                            interval_knowledge.append({
                                **knowledge_base,
                                'start_time': interval['start'],
                                'end_time': interval['end']
                            })
        
        # 批量获取标签
        labels = self.get_labels_batch(list(entities_to_label))
        
        # 为知识条目添加标签
        for knowledge in point_knowledge + interval_knowledge:
            knowledge['subject_label'] = labels.get(knowledge['subject'], knowledge['subject'])
            knowledge['predicate_label'] = labels.get(knowledge['predicate'], knowledge['predicate'])
            knowledge['object_label'] = labels.get(knowledge['object'], knowledge['object'])
        
        time.sleep(self.delay)
        return point_knowledge, interval_knowledge, new_entities
    
    def _extract_main_value_info(self, mainsnak):
        """提取主声明值信息"""
        if not mainsnak.get('datavalue'):
            return None, None
        
        datavalue = mainsnak['datavalue']
        value_type = datavalue.get('type', '')
        value = datavalue.get('value', {})
        
        if value_type == 'wikibase-entityid':
            return value.get('id', ''), 'entity'
        elif value_type == 'string':
            return value, 'string'
        elif value_type == 'time':
            time_value = value.get('time', '')
            if time_value:
                time_value = re.sub(r'^\+(.*)T.*', r'\1', time_value)
            return time_value, 'time'
        else:
            return str(value), 'other'
    
    def _extract_time_qualifiers(self, qualifiers):
        """提取时间限定符"""
        point_times = defaultdict(list)
        interval_times = []
        
        # 提取时间点限定符
        for time_prop in ['P585', 'P569', 'P570', 'P571', 'P576']:
            if time_prop in qualifiers:
                for snak in qualifiers[time_prop]:
                    if snak.get('datavalue') and snak['datavalue'].get('type') == 'time':
                        time_value = snak['datavalue']['value'].get('time', '')
                        if time_value:
                            time_value = re.sub(r'^\+(.*)T.*', r'\1', time_value)
                            point_times[time_prop].append(time_value)
        
        # 提取时间区间限定符
        start_times = []
        end_times = []
        
        if 'P580' in qualifiers:
            for snak in qualifiers['P580']:
                if snak.get('datavalue') and snak['datavalue'].get('type') == 'time':
                    time_value = snak['datavalue']['value'].get('time', '')
                    if time_value:
                        time_value = re.sub(r'^\+(.*)T.*', r'\1', time_value)
                        start_times.append(time_value)
        
        if 'P582' in qualifiers:
            for snak in qualifiers['P582']:
                if snak.get('datavalue') and snak['datavalue'].get('type') == 'time':
                    time_value = snak['datavalue']['value'].get('time', '')
                    if time_value:
                        time_value = re.sub(r'^\+(.*)T.*', r'\1', time_value)
                        end_times.append(time_value)
        
        # 构建时间区间对
        if start_times and end_times:
            # 简单匹配：取第一个开始时间和第一个结束时间
            interval_times.append({
                'start': start_times[0],
                'end': end_times[0]
            })
        elif start_times:
            for start_time in start_times:
                interval_times.append({
                    'start': start_time,
                    'end': None
                })
        elif end_times:
            for end_time in end_times:
                interval_times.append({
                    'start': None,
                    'end': end_time
                })
        
        return {
            'point_times': dict(point_times),
            'interval_times': interval_times
        }
    
    def analyze_entity_types(self, entity_ids):
        """分析实体类型，优先处理可能包含更多时间知识的实体"""
        if not entity_ids:
            return {}
        
        labels = self.get_labels_batch(entity_ids)
        
        # 时间相关关键词
        time_related_keywords = [
            'president', 'prime minister', 'king', 'queen', 'emperor', 'ruler',
            'office', 'position', 'event', 'war', 'battle', 'election',
            'organization', 'company', 'government', 'party',
            'period', 'era', 'age', 'century'
        ]
        
        type_scores = {}
        for entity_id in entity_ids:
            score = 0
            label = labels.get(entity_id, '').lower()
            
            for keyword in time_related_keywords:
                if keyword in label:
                    score += 1
            
            # 考虑实体作为客体的频率
            object_count = self.entity_priority.get(entity_id, 0)
            score += min(object_count, 5)
            
            type_scores[entity_id] = score
        
        return type_scores
    
    def process_batch_parallel(self, entity_batch):
        """并行处理实体批次"""
        return self.extract_temporal_knowledge_from_entities(entity_batch)
    
    def expand_knowledge_iteratively(self, initial_entities, target_knowledge_count=10000000, max_entities=500000):
        """
        迭代扩展时间知识
        
        参数:
            initial_entities: 初始实体集合
            target_knowledge_count: 目标知识数量
            max_entities: 最大实体数量限制
        """
        entity_set = set(initial_entities)
        iteration = 0
        
        print("开始迭代抽取Wikidata时间知识...")
        print(f"初始实体数: {len(initial_entities)}, 目标知识数: {target_knowledge_count}")
        
        with tqdm(total=target_knowledge_count, desc="抽取知识") as pbar:
            while (len(self.point_knowledge) + len(self.interval_knowledge) < target_knowledge_count and
                   len(entity_set) < max_entities):
                
                iteration += 1
                unprocessed = list(entity_set - self.processed_entities)
                
                if not unprocessed:
                    print("没有更多未处理实体")
                    break
                
                # 优先处理高价值实体
                if iteration > 1:
                    type_scores = self.analyze_entity_types(unprocessed)
                    unprocessed.sort(key=lambda e: type_scores.get(e, 0), reverse=True)
                else:
                    unprocessed.sort(key=lambda e: self.entity_priority.get(e, 0), reverse=True)
                
                # 分批处理
                batches = [unprocessed[i:i+self.batch_size] 
                          for i in range(0, len(unprocessed), self.batch_size)]
                
                total_new_knowledge = 0
                
                # 并行处理批次
                with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
                    future_to_batch = {
                        executor.submit(self.process_batch_parallel, batch): batch 
                        for batch in batches
                    }
                    
                    for future in concurrent.futures.as_completed(future_to_batch):
                        point_knowledge, interval_knowledge, new_entities = future.result()
                        
                        # 处理新知识
                        for knowledge in point_knowledge:
                            key = (knowledge['subject'], knowledge['predicate'], 
                                   knowledge['object'], knowledge.get('time_qualifier', ''), knowledge['time'])
                            if key not in self.knowledge_keys:
                                self.point_knowledge.append(knowledge)
                                self.knowledge_keys.add(key)
                                if knowledge['object'] not in self.entity_priority.keys():
                                    self.entity_priority[knowledge['object']] = 0
                                self.entity_priority[knowledge['object']] += 1
                        
                        for knowledge in interval_knowledge:
                            key = (knowledge['subject'], knowledge['predicate'], 
                                   knowledge['object'], knowledge.get('start_time'), knowledge.get('end_time'))
                            if key not in self.knowledge_keys:
                                self.interval_knowledge.append(knowledge)
                                self.knowledge_keys.add(key)
                                if knowledge['object'] not in self.entity_priority.keys():
                                    self.entity_priority[knowledge['object']] = 0
                                self.entity_priority[knowledge['object']] += 1
                        
                        # 更新实体集合
                        entity_set.update(new_entities)
                        total_new_knowledge += len(point_knowledge) + len(interval_knowledge)
                
                # 标记已处理
                self.processed_entities.update(unprocessed)
                self.remain_entites = entity_set
                
                # 更新进度条
                current_total = len(self.point_knowledge) + len(self.interval_knowledge)
                pbar.update(current_total - pbar.n)
                pbar.set_postfix({
                    '实体数': len(entity_set),
                    '已处理': len(self.processed_entities),
                    '时间点': len(self.point_knowledge),
                    '时间区间': len(self.interval_knowledge)
                })
                
                # 定期保存进度
                self.save_progress('wikidata_temporal_knowledge_progress.json')
                
                time.sleep(self.delay)
        
        return entity_set
    
    def save_results(self, output_file):
        """保存结果到JSON文件"""
        output = {
            'point_knowledge': self.point_knowledge,
            'interval_knowledge': self.interval_knowledge,
            'total_entities': len(self.processed_entities),
            'total_knowledge': len(self.point_knowledge) + len(self.interval_knowledge),
            'label_cache': dict(self.label_cache)
        }
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(output, f, ensure_ascii=False, indent=2)
        
        print(f"结果已保存到: {output_file}")
    
    def save_progress(self, progress_file):
        """保存进度"""
        progress = {
            'remain_entities': list(self.remain_entites),
            'processed_entities': list(self.processed_entities),
            'point_knowledge': self.point_knowledge,
            'interval_knowledge': self.interval_knowledge,
            'entity_priority': dict(self.entity_priority),
            'label_cache': dict(self.label_cache)
        }
        
        with open(progress_file, 'w', encoding='utf-8') as f:
            json.dump(progress, f, ensure_ascii=False, indent=2)

    def main(self):
        # 示例使用
        progress_file = json.load(open('wikidata_temporal_knowledge_progress.json'))
        initial_entities = progress_file['remain_entities']
        self.processed_entities = set(progress_file['processed_entities'])
        self.point_knowledge = progress_file['point_knowledge']
        self.interval_knowledge = progress_file['interval_knowledge']
        self.entity_priority = progress_file['entity_priority']
        self.label_cache = progress_file['label_cache']

        file = pd.read_csv(open('wikidata_temporal_properties.csv','r'))
        all_entities = set(file['head_entity_id'])
        initial_entities.extend(list(all_entities))
        
        '''
        try:
            progress_file = json.load(open('wikidata_temporal_knowledge_progress.json'))
            initial_entities = progress_file['remain_entities']
            self.processed_entities = set(progress_file['processed_entities'])
            self.point_knowledge = progress_file['point_knowledge']
            self.interval_knowledge = progress_file['interval_knowledge']
            self.entity_priority = progress_file['entity_priority']
            self.label_cache = progress_file['label_cache']

            if len(set(initial_entities) - self.processed_entities) == 0:
                
                extend_e_set = set()
                extend_e_file = open('entity2id.txt','r')
                for line in extend_e_file.readlines():
                    entity, _ = line.strip().split('\t')
                    extend_e_set.add(entity)
                initial_entities.extend(list(extend_e_set))
                
                #month = random.choice(['01','02','03','04','05','06','07','08','09','10','11','12'])
                #initial_entities.extend(list(get_hot_entities_alternative(k=1000, month=month)))
                initial_entities.extend(['Q649', 'Q956', 'Q61', 'Q84', 'Q7473516','Q18808','Q7156','Q8682','Q483020','Q9616','Q130521','Q130582','Q19317'])
        
        except:
            initial_entities = ['Q76', 'Q30', 'Q11696', 'Q22686', 'Q148', 'Q17', 'Q884', 'Q423', 'Q159', 'Q458', 'Q18756', 'Q11010', 'Q7318', 'Q881']  # 奥巴马、美国、美国总统、特朗普
            initial_entities.extend(list(get_hot_entities_alternative(k=1000, month='01')))
        '''
        # 执行抽取
        final_entities = self.expand_knowledge_iteratively(
            initial_entities, 
            target_knowledge_count=10000000,
            max_entities=500000
        )
        
        # 保存最终结果
        self.save_results('wikidata_temporal_knowledge_final.json')
        
        # 打印摘要
        print("\n抽取完成!")
        print(f"最终实体数量: {len(final_entities)}")
        print(f"时间点知识: {len(self.point_knowledge)}")
        print(f"时间区间知识: {len(self.interval_knowledge)}")
        print(f"总知识数量: {len(self.point_knowledge) + len(self.interval_knowledge)}")

if __name__ == "__main__":
    extractor = WikidataTemporalKnowledgeExtractor()
    extractor.main()
    
            

In [ ]:
import json
final_file = json.load(open('wikidata_temporal_knowledge_final.json'))
print(final_file.keys())
point_knowledge = final_file['point_knowledge']
interval_knowledge = final_file['interval_knowledge']

all_fact = set()
all_entity = set()
all_rel = set()
for f in point_knowledge:
    all_fact.add((f['subject'], f['predicate'], f['object'], f['time']))
    all_entity.add(f['subject'])
    all_entity.add(f['object'])
    all_rel.add(f['predicate'])
for f in interval_knowledge:
    all_fact.add((f['subject'], f['predicate'], f['object'], f['start_time'], f['end_time']))
    all_entity.add(f['subject'])
    all_entity.add(f['object'])
    all_rel.add(f['predicate'])
print(len(all_fact))
print(len(all_entity))
print(len(all_rel))

'''
initial_entities = progress_file['remain_entities']
processed_entities = set(progress_file['processed_entities'])
point_knowledge = progress_file['point_knowledge']
interval_knowledge = progress_file['interval_knowledge']
entity_priority = progress_file['entity_priority']
label_cache = progress_file['label_cache']

all_fact = set()
for f in point_knowledge:
    all_fact.add((f['subject'], f['predicate'], f['object'], f['time']))
for f in interval_knowledge:
    all_fact.add((f['subject'], f['predicate'], f['object'], f['start_time'], f['end_time']))
print(len(all_fact))
'''

In [ ]:
import re
import requests
import json
import csv
import time
import concurrent.futures
from tqdm import tqdm
from collections import defaultdict

class WikidataLabelDescFetcher:
    def __init__(self, delay=0.1, max_workers=10):
        """
        初始化Wikidata标签和描述获取器
        
        参数:
            delay: 请求间延迟（秒）
            max_workers: 最大并行工作线程数
        """
        self.delay = delay
        self.max_workers = max_workers
        self.headers = {
            'User-Agent': 'WikidataLabelDescFetcher/1.0 (https://example.com; contact@example.com)'
        }
        
        # 批量处理相关
        self.batch_size = 50  # Wikidata API每批最多50个实体
        
        # 结果存储
        self.result_dict = {}
        
    def get_labels_descriptions_batch(self, entity_ids, max_retries=3):
        """
        批量获取实体的标签和描述
        返回: 字典 {entity_id: [label, description]}
        """
        if not entity_ids:
            return {}
            
        # 过滤有效实体ID
        valid_ids = [eid for eid in entity_ids if eid.startswith('Q') or eid.startswith('P')]
        if not valid_ids:
            return {}
            
        # 将实体ID列表转换为逗号分隔的字符串
        ids_str = "|".join(valid_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en',
            'props': 'labels|descriptions'
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params, headers=self.headers, timeout=30)
                
                if response.status_code == 200:
                    data = response.json()
                    entities = data.get('entities', {})
                    
                    batch_result = {}
                    for eid, entity in entities.items():
                        # 获取英文标签
                        labels = entity.get('labels', {})
                        if 'en' in labels:
                            label = labels['en'].get('value', eid)
                        else:
                            # 如果没有英文标签，尝试其他语言或使用ID
                            label = eid
                            for lang in labels:
                                if labels[lang].get('value'):
                                    label = labels[lang].get('value')
                                    break
                        
                        # 获取英文描述
                        descriptions = entity.get('descriptions', {})
                        if 'en' in descriptions:
                            description = descriptions['en'].get('value', '')
                        else:
                            # 如果没有英文描述，尝试其他语言或使用空字符串
                            description = ''
                            for lang in descriptions:
                                if descriptions[lang].get('value'):
                                    description = descriptions[lang].get('value')
                                    break
                        
                        batch_result[eid] = [label, description]
                    
                    return batch_result
                    
                elif response.status_code == 429:
                    # 处理速率限制
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
                        
            except requests.exceptions.RequestException as e:
                print(f"请求异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(5)
        
        # 如果所有重试都失败，为这批实体创建默认结果
        default_result = {}
        for eid in valid_ids:
            default_result[eid] = [eid, '']  # 使用ID作为label，空描述
        return default_result
    
    def process_entities_parallel(self, entity_ids):
        """
        使用并行处理处理实体列表，返回结果字典
        """
        try:
            self.result_dict = dict(json.load(open('wikidata_labels_descriptions.json')))
        except:
            pass
        # 将实体ID分成批次
        batches = [entity_ids[i:i+self.batch_size] for i in range(0, len(entity_ids), self.batch_size)]
        
        print(f"将处理 {len(entity_ids)} 个实体，分为 {len(batches)} 批，每批 {self.batch_size} 个")
        print(f"使用 {self.max_workers} 个并行工作线程")
        
        # 使用线程池并行处理批次
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            # 提交所有批次任务
            future_to_batch = {executor.submit(self.get_labels_descriptions_batch, batch): batch 
                             for batch in batches}
            
            # 收集结果并显示进度
            for future in tqdm(concurrent.futures.as_completed(future_to_batch), 
                              total=len(batches), desc="获取标签和描述"):
                batch_result = future.result()
                self.result_dict.update(batch_result)
                
                # 添加延迟以避免过于频繁的请求
                time.sleep(self.delay)
                self.save_results('wikidata_labels_descriptions.json')
                self.save_results_csv('wikidata_labels_descriptions.csv')
        
        return self.result_dict
    
    def clean_description(self, description):
        """
        清理描述文本，移除时间相关的内容
        """
        self.time_patterns = [
            r'\(\s*\d{4}\s*[-–—]\s*\d{4}\s*\)',  # (1929-2003)
            r'\(\s*\d{4}\s*\)',  # (1929)
            r'from\s+\d{4}\s+to\s+\d{4}',  # from 1929 to 2003
            r'since\s+\d{4}\s+to\s+\d{4}',  # since 1929 to 2003
            r'\d{4}\s*[-–—]\s*\d{4}',  # 1929-2003 (无括号)
            r'born\s+\d{4}',  # born 1929
            r'\b\d{4}\s*-\s*\d{4}\b',  # 1929 - 2003
            r'\b\d{4}\s*–\s*\d{4}\b',  # 1929 – 2003 (不同种类的横线)
            r'\b\d{4}\s*—\s*\d{4}\b',  # 1929 — 2003
        ]

        if not description:
            return ""
        
        cleaned = description
        
        # 应用所有时间模式
        for pattern in self.time_patterns:
            cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE)
        
        # 清理多余的空格和标点
        cleaned = re.sub(r'\s+', ' ', cleaned)  # 多个空格变一个
        cleaned = re.sub(r'\s*,\s*,', ',', cleaned)  # 清理多余的逗号
        cleaned = re.sub(r'^\s*[,-]\s*', '', cleaned)  # 开头是逗号或横线
        cleaned = re.sub(r'\s*[,-]\s*$', '', cleaned)  # 结尾是逗号或横线
        cleaned = cleaned.strip()
        
        return cleaned.replace('()','')
    
    def save_results(self, output_file):
        """
        保存结果到JSON文件
        """
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(self.result_dict, f, ensure_ascii=False, indent=2)
        print(f"结果已保存到: {output_file}")
        print(f"共获取 {len(self.result_dict)} 个实体的标签和描述")
    
    def save_results_csv(self, output_file):
        """
        保存结果到CSV文件
        """
        with open(output_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['wikidata_id', 'label', 'description'])
            for entity_id, (label, description) in self.result_dict.items():
                writer.writerow([entity_id, label, self.clean_description(description)])
        print(f"CSV结果已保存到: {output_file}")

def load_entity_ids_from_file(file_path):
    """
    从文件加载实体ID列表
    """
    entity_ids = set()
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            csv_reader = csv.reader(f)
            
            for row in csv_reader:
                if row[0] == 'original_idx_text':
                    continue
                if len(row) > 2:
                    Q_id = row[2]
                    # 验证是否为有效的Wikidata ID格式
                    if Q_id and (Q_id.startswith('Q') and Q_id[1:].isdigit()):
                        entity_ids.add(Q_id)
        
        print(f"从文件加载了 {len(entity_ids)} 个有效的实体ID")
        return list(entity_ids)
        
    except Exception as e:
        print(f"加载实体ID时出错: {e}")
        return []


def main():
    # 从文件加载实体ID
    output_json = "wikidata_labels_descriptions.json"
    output_csv = "wikidata_labels_descriptions.csv"
    
    # 加载实体ID
    entity_ids = list(all_entity)
    
    if not entity_ids:
        print("没有找到有效的实体ID")
        return
    
    # 显示前几个实体ID作为示例
    print(f"示例实体ID: {entity_ids[:5]}")
    
    # 创建获取器并处理
    print("开始获取标签和描述...")
    fetcher = WikidataLabelDescFetcher(delay=0.1, max_workers=10)
    
    start_time = time.time()
    result_dict = fetcher.process_entities_parallel(entity_ids)
    end_time = time.time()
    
    # 保存结果
    fetcher.save_results(output_json)
    fetcher.save_results_csv(output_csv)
    
    # 打印统计信息
    print(f"\n处理完成!")
    print(f"总耗时: {end_time - start_time:.2f} 秒")
    print(f"成功获取: {len(result_dict)} 个实体")
    
    # 显示一些示例结果
    print("\n示例结果:")
    for i, (entity_id, (label, desc)) in enumerate(list(result_dict.items())[:3]):
        print(f"  {entity_id}:")
        print(f"    标签: {label}")
        print(f"    描述: {desc}")

if __name__ == "__main__":
    main()

In [ ]:
# 抓取新关系的label
import requests
import time
import sys

def get_property_labels(property_ids, language='en', delay=0.1, max_retries=3):
    """
    批量获取 Wikidata 属性标签
    
    参数:
        property_ids: Wikidata 属性 ID 列表 (例如 ["P31", "P21"])
        language: 需要的标签语言 (默认 'en')
        delay: 请求间延迟（秒），用于控制请求频率
        max_retries: 最大重试次数
    
    返回:
        字典: {property_id: label} 或 {property_id: None}（如果未找到）
    """
    headers = {
        'User-Agent': 'PropertyLabelFetcher/1.0 (https://example.com; contact@example.com)'
    }
    
    base_url = "https://www.wikidata.org/w/api.php"
    results = {}
    
    # 将属性ID列表分批次处理，每批50个
    batch_size = 50
    batches = [property_ids[i:i + batch_size] for i in range(0, len(property_ids), batch_size)]
    
    for batch in batches:
        params = {
            'action': 'wbgetentities',
            'ids': '|'.join(batch),
            'props': 'labels',
            'languages': language,
            'format': 'json'
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(base_url, params=params, headers=headers, timeout=30)
                if response.status_code == 200:
                    data = response.json()
                    
                    if 'entities' in data:
                        for prop_id in batch:
                            entity = data['entities'].get(prop_id, {})
                            labels = entity.get('labels', {})
                            
                            if language in labels:
                                results[prop_id] = labels[language]['value']
                            else:
                                results[prop_id] = None
                    
                    break  # 成功则跳出重试循环
                    
                elif response.status_code == 429:
                    # 遇到速率限制，等待更长时间
                    wait_time = delay * (2 ** attempt)  # 指数退避
                    print(f"速率限制，等待 {wait_time} 秒后重试...")
                    time.sleep(wait_time)
                else:
                    print(f"HTTP 错误 {response.status_code}，尝试 {attempt + 1}/{max_retries}")
                    if attempt < max_retries - 1:
                        time.sleep(delay)
                        
            except requests.exceptions.RequestException as e:
                print(f"请求异常: {e}，尝试 {attempt + 1}/{max_retries}")
                if attempt < max_retries - 1:
                    time.sleep(delay)
        
        # 批次间延迟
        time.sleep(delay)
    
    return results

def save_results(results, output_file='new_property_labels.csv'):
    """将结果保存到 CSV 文件"""
    import csv
    
    with open(output_file, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Property_ID', 'Label'])
        
        for prop_id, label in results.items():
            writer.writerow([prop_id, label if label else 'Not Found'])
    
    print(f"结果已保存到: {output_file}")

def main():
    # 示例属性 ID 列表 - 替换为您实际的属性 ID
    property_ids = list(all_rel)
    
    print(f"开始获取 {len(property_ids)} 个属性的标签...")
    
    # 获取属性标签
    results = get_property_labels(property_ids, language='en', delay=0.1)
    
    # 打印结果
    print("\n属性标签结果:")
    found_count = 0
    for prop_id, label in results.items():
        status = label if label else "未找到"
        print(f"{prop_id}: {status}")
        if label:
            found_count += 1
    
    print(f"\n成功获取 {found_count}/{len(property_ids)} 个属性的标签")
    
    # 保存到文件
    save_results(results)
    
    return results

if __name__ == "__main__":
    main()

In [ ]:
id_2_rel = {}
id_2_entity_label_description = {}
label_2_des_dict = {}
delete_descriptions = ['family name', 'doctoral thesis', 'scholarly article', 'scientific article', 'episode of', 'studio album', 'Journal article']
csv_reader = csv.reader(open("new_property_labels.csv"))
for row in csv_reader:
	if row[0] == 'Property_ID':
		continue
	id_2_rel[row[0]] = row[1]

csv_reader = csv.reader(open("wikidata_labels_descriptions.csv"))
for row in csv_reader:
	if row[0] == 'wikidata_id':
		continue
	read_entity_idx = row[0]
	read_entity_label = row[1]
	read_entity_des = row[2]
	if read_entity_idx == read_entity_label:
		continue
	if read_entity_idx not in id_2_entity_label_description.keys():
		if read_entity_des != '':
			id_2_entity_label_description[read_entity_idx] = [read_entity_label, read_entity_des]
		else:
			id_2_entity_label_description[read_entity_idx] = [read_entity_label, read_entity_label]
	
	if read_entity_label not in label_2_des_dict.keys():
		if read_entity_des != '':
			label_2_des_dict[read_entity_label] = read_entity_des
		else:
			label_2_des_dict[read_entity_label] = read_entity_label

In [ ]:
import json
final_file = json.load(open('wikidata_temporal_knowledge_final.json'))
print(final_file.keys())
point_knowledge = final_file['point_knowledge']
interval_knowledge = final_file['interval_knowledge']
cached_labels = dict(final_file['label_cache'])
max_time = '0000-00-00'
min_time = '2025-12-30'


e_2_ponit_fact = {}
e_2_span_fact = {}

all_fact = set()
point_fact = set()
span_fact = set()
all_entity = set()
all_rel = set()
no_cached_rels = set()
for f in point_knowledge:

    time = f['time']
    if f['time'] < '1800-00-00':
        time = '1800-00-00'
    if f['time'] > '2025-10-28':
        time = '2025-10-28'
    
    if f['subject'] not in id_2_entity_label_description.keys() or f['object'] not in id_2_entity_label_description.keys():
        continue

    s, r, o = id_2_entity_label_description[f['subject']][0], id_2_rel[f['predicate']], id_2_entity_label_description[f['object']][0]

    all_fact.add((s, r, o, time))
    all_entity.add(s)
    all_entity.add(o)
    all_rel.add(r)
    point_fact.add((s, r, o, time))

    if s not in e_2_ponit_fact.keys():
        e_2_ponit_fact[s] = set()
    e_2_ponit_fact[s].add((s, r, o, time))

    if o not in e_2_ponit_fact.keys():
        e_2_ponit_fact[o] = set()
    e_2_ponit_fact[o].add((s, r, o, time))

    if min_time > time:
        min_time = time
    if max_time < time:
        max_time = time

for f in interval_knowledge:

    start_time = f['start_time']
    end_time = f['end_time']
    
    if start_time == None or end_time == None:
        continue
    
    if f['start_time'] < '1800-00-00':
        start_time = '1800-00-00'
    if f['end_time'] > '2025-10-28':
        end_time = '2025-10-28'

    if f['end_time'] < '1800-00-00':
       continue
    if f['start_time'] > '2025-10-28':
        continue

    if f['end_time'] < f['start_time']:
        continue

    if f['subject'] not in id_2_entity_label_description.keys() or f['object'] not in id_2_entity_label_description.keys():
        continue

    s, r, o = id_2_entity_label_description[f['subject']][0], id_2_rel[f['predicate']], id_2_entity_label_description[f['object']][0]

    all_fact.add((s, r, o, start_time, end_time))
    all_entity.add(s)
    all_entity.add(r)
    all_rel.add(o)
    span_fact.add((s, r, o, start_time, end_time))

    if s not in e_2_span_fact.keys():
        e_2_span_fact[s] = set()
    e_2_span_fact[s].add((s, r, o, start_time, end_time))

    if o not in e_2_span_fact.keys():
        e_2_span_fact[o] = set()
    e_2_span_fact[o].add((s, r, o, start_time, end_time))

    if start_time != None and min_time > start_time:
        min_time = start_time
    if end_time != None and max_time < end_time:
        max_time = end_time

print(len(all_fact))
print(len(all_entity))
print(len(all_rel))
print([min_time, max_time])
print([len(point_fact), len(span_fact)])

only_point_e = set()
only_span_e = set()
point_span_e = set()
average_fact_num = 0
point_span_e_average_point_fact_num = 0
point_span_e_average_span_fact_num = 0
for e in all_entity:
    if e in e_2_span_fact.keys() and e in e_2_ponit_fact.keys():
        point_span_e.add(e)
        point_span_e_average_point_fact_num += len(e_2_ponit_fact[e])
        point_span_e_average_span_fact_num += len(e_2_span_fact[e])
    if e in e_2_span_fact.keys():
        average_fact_num += len(e_2_span_fact[e])
    if e in e_2_ponit_fact.keys():
        average_fact_num += len(e_2_ponit_fact[e])
    
    if e in e_2_span_fact.keys() and e not in e_2_ponit_fact.keys():
        only_span_e.add(e)
    
    if e not in e_2_span_fact.keys() and e in e_2_ponit_fact.keys():
        only_point_e.add(e)

print([len(point_span_e), len(only_point_e), len(only_span_e)])
print(float(average_fact_num)/float(len(all_entity)))
print(float(point_span_e_average_point_fact_num)/float(len(point_span_e)))
print(float(point_span_e_average_span_fact_num)/float(len(point_span_e)))

In [ ]:
#check
all_knowledge = []

for f in point_fact:
    #print(f)
    if f[0] not in label_2_des_dict.keys() or f[2] not in label_2_des_dict.keys():
        print("entity error: "+str(f))
    all_knowledge.append(f)

for f in span_fact:
    if f[0] not in label_2_des_dict.keys() or f[2] not in label_2_des_dict.keys():
        print("entity error: "+str(f))
    if f[-2] > f[-1]:
        print("time error: "+str(f))
        continue
    all_knowledge.append(f)

print(len(all_knowledge))

In [ ]:
import pickle
pickle.dump([label_2_des_dict, all_knowledge], open('Wiki_e12k_f500k.pkl', 'wb'))

In [ ]:
node_2_keys, fact = pickle.load(open('Wiki_e12k_f500k.pkl', 'rb'))
print([(key, node_2_keys[key]) for key in list(node_2_keys.keys())[:30]])
print(list(fact)[:30])
print(list(fact)[-30:])

In [ ]:
import pickle
node_2_keys, fact = pickle.load(open('Wiki_e12k_f500k.pkl', 'rb'))
print([(key, node_2_keys[key]) for key in list(node_2_keys.keys())[:30]])
print(list(fact)[:30])

In [ ]:
import matplotlib.pyplot as plt
import tqdm
from datetime import date, timedelta

# split train valid test
# 首先确定切分时间戳
all_times = set()
t_2_stamp_fact = {}
t_2_span_fact = {}
t_2_start = {}
t_2_end = {}
new_fact = set()
for f in tqdm.tqdm(fact):
    if len(f) == 4:
        t = int(f[-1].split('-')[0])
        if t > 2025:
            continue
        all_times.add(t)
        if t not in t_2_stamp_fact.keys():
            t_2_stamp_fact[t] = set()
        t_2_stamp_fact[t].add((f[0],f[1],f[2],t))
        new_fact.add((f[0],f[1],f[2],t))
    if len(f) == 5:
        t_s = int(f[-2].split('-')[0])
        t_e = int(f[-1].split('-')[0])
        if t_s > 2025 or t_e > 2025:
            continue
        if t_s > t_e:
            continue
        if t_s == t_e:
            all_times.add(t_s)
            if t_s not in t_2_stamp_fact.keys():
                t_2_stamp_fact[t_s] = set()
            t_2_stamp_fact[t_s].add((f[0],f[1],f[2],t_s))
            new_fact.add((f[0],f[1],f[2],t_s))
        else:
            all_times.add(t_s)
            all_times.add(t_e)
            for t in range(t_s, t_e+1):
                if t not in t_2_span_fact.keys():
                    t_2_span_fact[t] = set()
                t_2_span_fact[t].add((f[0], f[1], f[2], t_s, t_e))
            
            if t_s not in t_2_start.keys():
                t_2_start[t_s] = set()
            t_2_start[t_s].add((f[0], f[1], f[2], t_s, t_e))

            if t_e not in t_2_end.keys():
                t_2_end[t_e] = set()
            t_2_end[t_e].add((f[0], f[1], f[2], t_s, t_e))

            new_fact.add((f[0], f[1], f[2], t_s, t_e))


sorted_all_times = sorted(list(all_times))
sorted_stamp_nums = []
sorted_span_nums = []
sorted_start_nums = []
sorted_end_nums = []
for t in sorted_all_times:
    if t in t_2_stamp_fact.keys():
        sorted_stamp_nums.append(len(t_2_stamp_fact[t]))
    else:
        sorted_stamp_nums.append(0)
    
    if t in t_2_span_fact.keys():
        sorted_span_nums.append(len(t_2_span_fact[t]))
    else:
        sorted_span_nums.append(0)

    if t in t_2_start.keys():
        sorted_start_nums.append(len(t_2_start[t]))
    else:
        sorted_start_nums.append(0)
    
    if t in t_2_end.keys():
        sorted_end_nums.append(len(t_2_end[t]))
    else:
        sorted_end_nums.append(0)
print(sorted_all_times)
print(len(sorted_all_times))
print(sorted_stamp_nums)
print(len(sorted_stamp_nums))
print(sorted_span_nums)
print(len(sorted_span_nums))


In [ ]:
split_ratio = 0.85
valid_preiod = 0.025 
# 切分时间戳
print('train/valid/test split point')
print([int(split_ratio*len(sorted_all_times)), sorted_all_times[int(split_ratio*len(sorted_all_times))]])
print([int((split_ratio+valid_preiod)*len(sorted_all_times)), sorted_all_times[int((split_ratio+valid_preiod)*len(sorted_all_times))]])

print('train stamp span fact num')
# 切分后的训练集时间戳知识数量
print(sum(sorted_stamp_nums[:int(split_ratio*len(sorted_all_times))]))
# 切分后的训练集时间区间知识数量
print(sum(sorted_span_nums[:int(split_ratio*len(sorted_all_times))]))

print('valid stamp span fact num')
# 切分后的验证集时间戳知识数量
print(sum(sorted_stamp_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))
# 切分后的验证集时间区间知识数量
print(sum(sorted_span_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))

print('test stamp span fact num')
# 切分后的测试集时间戳知识数量
print(sum(sorted_stamp_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):]))
# 切分后的测试集时间区间知识数量
print(sum(sorted_span_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):]))

# 检查验证集和测试集各有多少时间区间知识结束和时间区间知识开始
print('train start end fact num')
print(sum(sorted_start_nums[:int(split_ratio*len(sorted_all_times))]))
print(sum(sorted_end_nums[:int(split_ratio*len(sorted_all_times))]))
print('valid start end fact num')
print(sum(sorted_start_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))
print(sum(sorted_end_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))

print('test start end fact num')
print(sum(sorted_start_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):-1]))
print(sum(sorted_end_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):-1]))

plt.plot(sorted_stamp_nums, color='skyblue', alpha=0.5)
plt.plot(sorted_span_nums, color='red', alpha=0.5)
plt.show()

In [ ]:
# 正式切分数据集
train_set = set()
valid_set = set()
test_set = set()

train_split_time = sorted_all_times[int(split_ratio*len(sorted_all_times))-1]
valid_split_time = sorted_all_times[int((split_ratio+valid_preiod)*len(sorted_all_times))-1]
print([train_split_time, valid_split_time])
for f in new_fact:
    if len(f) == 4: # stamp knowledge
        s, r, o, t = f
        if t <= train_split_time:
            train_set.add(f)
        elif t <= valid_split_time:
            valid_set.add(f)
        else:
            test_set.add(f)
    if len(f) == 5: # span knowledge
        s, r, o, t_s, t_o = f
        if t_s <= train_split_time:
            train_set.add(f)
        elif t_s <= valid_split_time:
            valid_set.add(f)
        else:
            test_set.add(f)
        
        if t_o > train_split_time:
            valid_set.add(f)
        if t_o > valid_split_time:
            test_set.add(f)

print([len(train_set), len(valid_set), len(test_set)])
print(len(train_set|valid_set|test_set))

# check the consistency
print('train stamp span fact num')
# 切分后的训练集时间戳知识数量
print(len([f for f in train_set if len(f) == 4]))
# 切分后的训练集时间区间知识数量
print(len([f for f in train_set if len(f) == 5]))

print('valid stamp span fact num')
# 切分后的训练集时间戳知识数量
print(len([f for f in valid_set if len(f) == 4]))
# 切分后的训练集时间区间知识数量
print(len([f for f in valid_set if len(f) == 5]))

print('test stamp span fact num')
# 切分后的训练集时间戳知识数量
print(len([f for f in test_set if len(f) == 4]))
# 切分后的训练集时间区间知识数量
print(len([f for f in test_set if len(f) == 5]))

# 检查验证集和测试集各有多少时间区间知识结束和时间区间知识开始
print('train start end fact num')
start_num, end_num = 0, 0
train_start_fact = set()
train_end_fact = set()
for f in train_set:
    if len(f) == 5:
        if f[-2] <= train_split_time:
            start_num += 1
            train_start_fact.add(f)
        if f[-1] <= train_split_time:
            end_num += 1
            train_end_fact.add(f)
print([start_num, end_num])

print('valid start end fact num')
start_num, end_num = 0, 0
valid_start_fact = set()
valid_end_fact = set()
for f in valid_set:
    if len(f) == 5:
        if f[-2] <= valid_split_time and f[-2] > train_split_time:
            start_num += 1
            valid_start_fact.add(f)
        if f[-1] <= valid_split_time and f[-1] > train_split_time:
            end_num += 1
            valid_end_fact.add(f)          
print([start_num, end_num])

print('test start end fact num')
start_num, end_num = 0, 0
test_start_fact = set()
test_end_fact = set()
for f in test_set:
    if len(f) == 5:
        if f[-2] > valid_split_time:
            start_num += 1
            test_start_fact.add(f)
        if f[-1] < sorted_all_times[-1] and f[-1] > valid_split_time:
            end_num += 1
            test_end_fact.add(f)
print([start_num, end_num])

# check train_valid_test 泄漏
# stamp knowledge
train_stamp_facts = set([f for f in train_set if len(f) == 4])
valid_stamp_facts = set([f for f in valid_set if len(f) == 4])
test_stamp_facts = set([f for f in test_set if len(f) == 4])

print("stamp train & valid")
print(len(train_stamp_facts&valid_stamp_facts))

print("stamp valid & test")
print(len(test_stamp_facts&valid_stamp_facts))

print("stamp train & test")
print(len(train_stamp_facts&test_stamp_facts))

# span knowledge
print("start train & valid")
print(len(train_start_fact&valid_start_fact))

print("start train & test")
print(len(train_start_fact&test_start_fact))

print("start test & valid")
print(len(test_start_fact&valid_start_fact))

print("end train & valid")
print(len(train_end_fact&valid_end_fact))

print("end train & test")
print(len(train_end_fact&test_end_fact))

print("end test & valid")
print(len(test_end_fact&valid_end_fact))

In [ ]:
pickle.dump([train_split_time, valid_split_time, train_set, valid_set, test_set, node_2_keys], open('Wiki_e12k_f500k_train_valid_test.pkl', 'wb'))